# Activité Pratique N°4 — Architecture Transformer
## Partie 2 — Exercice 3 : Mini-BERT

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Entraîner un Transformer bidirectionnel à retrouver des mots masqués.

**Objectif :** entraîner un Transformer **bidirectionnel** à reconstruire des mots masqués grâce au contexte à gauche **et** à droite (modélisation du langage masqué).

In [ ]:
import copy
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


def fixer_graine(graine=42):
    random.seed(graine)
    np.random.seed(graine)
    torch.manual_seed(graine)


def nb_parametres(modele):
    return sum(p.numel() for p in modele.parameters())


fixer_graine()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| appareil :", device)

## Corpus et vocabulaire

Petit corpus de phrases simples, générées à partir de modèles : chaque métier va avec ses actions, ses objets et son lieu.
Le mot masqué peut donc souvent être deviné **grâce au contexte de droite** (« boulangerie » permet de retrouver « boulanger »).

In [ ]:
fixer_graine()
METIERS = {
    "boulanger": (["prépare", "vend"], ["le pain", "les croissants", "la brioche"], "à la boulangerie"),
    "médecin": (["soigne", "examine"], ["les malades", "les enfants", "les patients"], "à l'hôpital"),
    "professeur": (["explique", "corrige"], ["la leçon", "les devoirs", "les examens"], "à l'école"),
    "pêcheur": (["attrape", "vend"], ["les poissons", "les sardines", "les crevettes"], "au port"),
    "fermier": (["récolte", "plante"], ["le blé", "les tomates", "les olives"], "à la ferme"),
    "mécanicien": (["répare", "nettoie"], ["la voiture", "le moteur", "les freins"], "au garage"),
    "cuisinier": (["prépare", "goûte"], ["la soupe", "le tajine", "le couscous"], "au restaurant"),
    "jardinier": (["arrose", "taille"], ["les fleurs", "les arbres", "les rosiers"], "au jardin"),
    "pilote": (["conduit", "pose"], ["l'avion", "l'appareil", "le gros porteur"], "à l'aéroport"),
    "libraire": (["range", "vend"], ["les livres", "les romans", "les magazines"], "à la librairie"),
    "pharmacien": (["vend", "prépare"], ["les médicaments", "les sirops", "les pommades"], "à la pharmacie"),
    "informaticien": (["programme", "teste"], ["le logiciel", "le serveur", "le site"], "au bureau"),
}
DETERMINANTS = ["le", "ce", "notre"]
MOMENTS = ["le matin", "le soir", "chaque jour", "le dimanche", "en hiver"]

phrases = [f"{det} {metier} {verbe} {objet} {lieu} {moment}"
           for metier, (verbes, objets, lieu) in METIERS.items()
           for det in DETERMINANTS for verbe in verbes for objet in objets for moment in MOMENTS]
random.shuffle(phrases)
coupe = int(0.85 * len(phrases))
phrases_train, phrases_val = phrases[:coupe], phrases[coupe:]
print(f"{len(phrases)} phrases : {len(phrases_train)} train, {len(phrases_val)} validation")
print("Exemples :", *phrases[:3], sep="\n  ")

In [ ]:
PAD, UNK, CLS, SEP, MASK = 0, 1, 2, 3, 4
SPECIAUX = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
LONGUEUR = 32

# Vocabulaire construit sur le corpus d'entraînement uniquement
mots = sorted({m for p in phrases_train for m in p.split()})
vocab = {t: i for i, t in enumerate(SPECIAUX + mots)}
inv_vocab = {i: t for t, i in vocab.items()}
V = len(vocab)


def encoder(phrase):
    ids = [CLS] + [vocab.get(m, UNK) for m in phrase.split()] + [SEP]
    return ids + [PAD] * (LONGUEUR - len(ids))


ids_train = torch.tensor([encoder(p) for p in phrases_train])
ids_val = torch.tensor([encoder(p) for p in phrases_val])
print("Vocabulaire :", V, "tokens (dont 5 spéciaux)")
print("Phrase encodée :", " ".join(inv_vocab[int(i)] for i in ids_train[0][:16]), "…")
print("Formes :", tuple(ids_train.shape), tuple(ids_val.shape))

## Masquage façon BERT

On sélectionne **15 %** des tokens ordinaires (jamais les tokens spéciaux), avec **au moins une position par phrase**. Parmi les positions sélectionnées :
**80 %** → `[MASK]`, **10 %** → token ordinaire aléatoire, **10 %** → inchangé.
La perte ne porte que sur les positions sélectionnées (`labels = -100` ailleurs).

In [ ]:
def masquer(input_ids, generateur=None):
    ordinaires = input_ids > MASK
    selection = (torch.rand(input_ids.shape, generator=generateur) < 0.15) & ordinaires
    for ligne in torch.nonzero(selection.sum(1) == 0).flatten():     # au moins une position par phrase
        candidats = torch.nonzero(ordinaires[ligne]).flatten()
        selection[ligne, candidats[torch.randint(len(candidats), (1,), generator=generateur)]] = True

    corrompu = input_ids.clone()
    r = torch.rand(input_ids.shape, generator=generateur)
    corrompu[selection & (r < 0.8)] = MASK
    aleatoire = selection & (r >= 0.8) & (r < 0.9)
    corrompu[aleatoire] = torch.randint(MASK + 1, V, (int(aleatoire.sum()),), generator=generateur)
    # r >= 0.9 : 10 % laissés inchangés

    labels = input_ids.clone()
    labels[~selection] = -100
    return corrompu, labels, selection


c, l, sel = masquer(ids_train[:2])
for k in range(2):
    print("Original :", " ".join(inv_vocab[int(i)] for i in ids_train[k] if i != PAD))
    print("Masqué   :", " ".join(inv_vocab[int(i)] for i in c[k] if i != PAD))
print(f"\nProportion sélectionnée : {masquer(ids_train)[2].sum() / (ids_train > MASK).sum():.1%}")

## Modèle

Embeddings de tokens et de positions de dimension 64, **2 couches d'encodeur à 4 têtes**, attention **bidirectionnelle** (sans masque causal),
**masque de padding** (`True` sur les `[PAD]`) et projection finale vers le vocabulaire.

In [ ]:
class MiniBERT(nn.Module):
    def __init__(self, vocabulaire, d_model=64, nhead=4, couches=2, longueur_max=LONGUEUR, dropout=0.1):
        super().__init__()
        self.tokens = nn.Embedding(vocabulaire, d_model, padding_idx=PAD)
        self.positions = nn.Embedding(longueur_max, d_model)
        self.norme = nn.LayerNorm(d_model)
        couche = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=4 * d_model, dropout=dropout,
                                            activation="gelu", batch_first=True)
        self.encodeur = nn.TransformerEncoder(couche, num_layers=couches, enable_nested_tensor=False)
        self.tete = nn.Linear(d_model, vocabulaire)

    def forward(self, ids, masque_padding=None):
        if masque_padding is None:
            masque_padding = ids == PAD
        x = self.tokens(ids) + self.positions(torch.arange(ids.shape[1], device=ids.device))
        x = self.encodeur(self.norme(x), src_key_padding_mask=masque_padding)
        return self.tete(x)                                     # (B, 32, V)


print("Paramètres :", nb_parametres(MiniBERT(V)))

## Entraînement et évaluation

Masquage **refait à chaque lot** (masquage dynamique). Sur la validation, masquage fixe (graine) pour comparer les époques.
Calcul des cibles : `labels[~positions_selectionnees] = -100` et `CrossEntropyLoss(ignore_index=-100)`.

In [ ]:
critere = nn.CrossEntropyLoss(ignore_index=-100)
val_entrees, val_labels, val_selection = masquer(ids_val, torch.Generator().manual_seed(123))


@torch.no_grad()
def exactitude_validation(modele):
    modele.eval()
    predits = modele(val_entrees.to(device)).cpu().argmax(-1)
    return (predits[val_selection] == ids_val[val_selection]).float().mean().item()


fixer_graine()
bert = MiniBERT(V).to(device)
optimiseur = torch.optim.AdamW(bert.parameters(), lr=2e-3)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(ids_train), batch_size=64, shuffle=True)
for epoch in range(1, 41):
    bert.train()
    total, nb = 0.0, 0
    for (ids,) in loader:
        entrees, labels, _ = masquer(ids)
        logits = bert(entrees.to(device))
        loss = critere(logits.reshape(-1, V), labels.to(device).reshape(-1))
        optimiseur.zero_grad()
        loss.backward()
        optimiseur.step()
        total, nb = total + loss.item(), nb + 1
    if epoch % 5 == 0:
        print(f"Époque {epoch:2d} | perte {total / nb:.3f} | exactitude validation {exactitude_validation(bert):.1%}")

### Les trois propositions les plus probables pour quelques mots masqués

In [ ]:
@torch.no_grad()
def top3(phrase, mot_cache):
    bert.eval()
    mots_phrase = phrase.split()
    position = mots_phrase.index(mot_cache) + 1                 # +1 pour [CLS]
    ids = torch.tensor([encoder(phrase)])
    ids[0, position] = MASK
    probas = torch.softmax(bert(ids.to(device))[0, position], -1).cpu()
    top = torch.topk(probas, 3)
    texte_masque = " ".join("[MASK]" if k == position - 1 else m for k, m in enumerate(mots_phrase))
    propositions = ", ".join(f"{inv_vocab[int(i)]} ({p:.0%})" for p, i in zip(top.values, top.indices))
    print(f"{texte_masque}\n   attendu : {mot_cache:14s} -> {propositions}")


top3(phrases_val[0], phrases_val[0].split()[1])     # le métier, deviné grâce au contexte de droite
top3(phrases_val[1], phrases_val[1].split()[2])     # le verbe
top3(phrases_val[2], phrases_val[2].split()[-1])    # fin du moment (peu prévisible)
top3(phrases_val[3], phrases_val[3].split()[1])

## Vérification essentielle : le padding n'influence pas les prédictions

On change les valeurs aux positions `[PAD]` en gardant **le même masque de padding** : en mode `eval()`, les prédictions des tokens utiles ne doivent pas changer.

In [ ]:
bert.eval()
ids = val_entrees[:8].to(device)
masque_padding = ids == PAD
ids_modifies = ids.clone()
ids_modifies[masque_padding] = torch.randint(MASK + 1, V, (int(masque_padding.sum()),), device=device)
with torch.no_grad():
    l1, l2 = bert(ids, masque_padding), bert(ids_modifies, masque_padding)
identiques = torch.allclose(l1[~masque_padding], l2[~masque_padding], atol=1e-5)
print("Prédictions des tokens utiles identiques :", identiques)
assert identiques

## Questions

**Pourquoi BERT peut-il utiliser le contexte à droite du mot masqué ?**
Son encodeur n'a **pas de masque causal** : chaque position voit toutes les autres. Ce n'est pas de la triche, car le mot à retrouver est **caché dans l'entrée** (`[MASK]`) : le contexte de droite ne contient pas la réponse.
C'est ce qui permet ici de retrouver « boulanger » grâce à « pain » et « boulangerie », placés après lui.

**Pourquoi calculer la perte uniquement sur les positions sélectionnées ?**
Aux autres positions l'entrée contient déjà la réponse : la prédire reviendrait à recopier, une tâche triviale qui dominerait la perte (85 % des tokens) sans rien apprendre.
On garde aussi les 10 % de positions sélectionnées **laissées inchangées** : le modèle ne sait pas quelles positions sont évaluées, il doit donc bien représenter **chaque** mot.

**Pourquoi ce modèle ne génère-t-il pas directement du texte comme GPT ?**
Il n'apprend pas à prédire le mot **suivant** à partir du passé, mais à combler un trou en voyant les deux côtés, en supposant connus la longueur et le reste de la phrase.
Pour générer de gauche à droite, il n'a pas de contexte de droite ; on peut remplir des masques de façon itérative, mais c'est lent et moins naturel que la génération autorégressive de GPT.